# LoRA: LOW-RANK ADAPTATION OF LARGE LANGUAGE MODELS

![LoRA: LOW-RANK ADAPTATION OF LARGE LANGUAGE MODELS - Abstract](assets/paper.png)

## 1. Background

Large language models such as GPT, LLaMA and other LLMs are built with billions of parameters. These parameters are the values the model learns during training, and they allow the model to understand and generate language. A common way to adapt a pretrained model to a new task is full fine-tuning, which updates every parameter in the model using task-specific training data.

This can work well, but it creates several practical problems. Large models need a huge amount of GPU memory during training, and they also require extra memory for optimizer states, which makes training even more expensive. Each fine-tuned version of the model must store a full new set of weights, so checkpoint files become large. As models are getting larger and larger with more parameters and tokens, this problem increases to a critical point hard to ignore.

LoRA stands for **Low-Rank Adaptation**. It was introduced as a more efficient way to adapt large pretrained models. Instead of changing all of the original model weights, LoRA keeps the pretrained weights frozen and learns only a small update. This update is represented using two low-rank matrices, which together approximate the changes needed for the new task. Because the rank is small, the number of trainable parameters is much lower than in full fine-tuning.

This design gives some advantages. Training becomes cheaper because only a small number of parameters need to be updated, memory usage is reduced, and the final task-specific checkpoint is much smaller, which makes it easier to save, share, and deploy. Finally, the original pretrained model stays unchanged, so the same base model can support many different LoRA adapters for different tasks.

## 2. Problem Statement

LoRA's central claim is that the *update* $\Delta W$ learned during adaptation has low intrinsic rank, so it is wasteful to represent $\Delta W$ as a full matrix. Instead, LoRA factorizes it as $\Delta W = B A$ with $B \in \mathbb{R}^{d\times r}$, $A \in \mathbb{R}^{r\times k}$ and $r \ll \min(d, k)$. The pretrained $W_0$ is frozen; only $B$ and $A$ are trained.

Suppose we have a pretrained model $P_{\Phi_0}(y \mid x)$ with parameters $\Phi_0$, and we want to adapt it to a downstream dataset $\mathcal{Z} = \{(x_i, y_i)\}_{i=1}^N$. 

**Full fine-tuning** searches for a new parameter vector $\Phi_0 + \Delta\Phi$ by maximizing

$$
\max_{\Phi}\; \sum_{(x, y) \in \mathcal{Z}} \sum_{t=1}^{|y|} \log P_{\Phi}(y_t \mid x, y_{<t})
$$

and the update $\Delta\Phi$ has the same dimension as $\Phi_0$.

**Parameter-efficient adaptation** factorizes the update through a much smaller parameter set $\Theta$ with $|\Theta| \ll |\Phi_0|$:

$$
\Delta\Phi \;=\; \Delta\Phi(\Theta), \qquad
\max_{\Theta}\; \sum_{(x, y) \in \mathcal{Z}} \sum_{t=1}^{|y|} \log P_{\Phi_0 + \Delta\Phi(\Theta)}(y_t \mid x, y_{<t}).
$$

LoRA is one particular choice of the encoding $\Delta\Phi(\Theta)$: for each targeted pretrained matrix $W_0 \in \mathbb{R}^{d \times k}$ the update is

$$
\Delta W \;=\; B A, \qquad B \in \mathbb{R}^{d \times r}, \;\; A \in \mathbb{R}^{r \times k}, \;\; r \ll \min(d, k).
$$

At inference, the model sees a single effective matrix $W = W_0 + B A$, so there is no architectural overhead — more on that shortly.

The schematic below shows the two computational paths (frozen base and trainable low-rank update) that are combined at the output.

![Lora Image](assets/lora.png)

[2] 


## 3. Why Existing Solutions Are Not Enough

Before LoRA, two broad families of parameter-efficient adaptation were already well known. The paper argues that each has a drawback LoRA is designed to avoid.

**Adapter layers**. Insert small bottleneck MLP modules between existing Transformer sublayers and train only those. They have few parameters, but they sit serially in the forward pass. In latency-sensitive, small-batch inference, even a few extra sequential matmuls add measurable latency — the paper reports up to ~20–30% slowdown on GPT-2 medium at batch size 1.

**Prompt / prefix tuning**. Prepend a learnable sequence of "soft" tokens to the input and train only those embeddings (or the activations they induce at each layer). This reserves part of the context window for adaptation, shortening the effective input available to the downstream task, and the paper reports that prefix tuning is difficult to optimize and that performance is non-monotone in the number of prefix tokens.

## 4. LoRA

For a targeted pretrained matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA represents the adaptation update as

$$
\Delta W \;=\; B A, \qquad B \in \mathbb{R}^{d \times r}, \;\; A \in \mathbb{R}^{r \times k}, \;\; r \ll \min(d, k).
$$

The effective weight used in the forward pass is $W = W_0 + \Delta W = W_0 + B A$. For an input $x$,

$$
h \;=\; W_0\, x \;+\; \frac{\alpha}{r}\, B A\, x .
$$

Four design choices from the paper are worth pausing on:

- **Initialization.** $A$ is initialized with a Gaussian, and $B$ is initialized to zero. At step 0, $\Delta W = B A = 0$, so the model starts exactly at the pretrained function — a safe starting point.
- **Scaling $\alpha / r$.** The update is multiplied by $\alpha/r$. This makes training roughly rank-agnostic: when we change $r$ we do not need to re-tune the learning rate. The paper sets $\alpha$ to the first $r$ they try and leaves it there.
- **Why small $r$?** The paper's hypothesis (supported empirically later in the paper) is that the task-specific update lives in a very low-dimensional subspace. The extreme result from the paper's Table 6 is that $r = 1$ or $r = 2$ is often enough for GPT-3 adaptation.
- **No extra parameters in the base path.** $W_0$ is frozen. No optimizer state is allocated for it, which is where the memory savings come from.

Below is a matrix-shape view of the factorization.

![Lora factorization](assets/factorization.png)
